# poppy_rs vs poppy (numpy): PSF Comparison
Computes the same PSF through both backends, checks numerical agreement, and compares timing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import time
import poppy
import poppy_rs

print(f'poppy version : {poppy.__version__}')
print(f'poppy_rs available: {poppy.accel_math._RUST_AVAILABLE}')
print(f'poppy_rs active   : {poppy.accel_math._USE_RUST}')

## 1. Simple circular aperture PSF
The simplest case: a circular pupil propagated to a detector via MFT.

In [ ]:
def make_optical_system(oversample=4, fov_arcsec=3.0):
    osys = poppy.OpticalSystem(oversample=oversample)
    osys.add_pupil(poppy.CircularAperture(radius=3.25))  # JWST-like 6.5m diameter
    osys.add_detector(pixelscale=0.063, fov_arcsec=fov_arcsec)  # NIRCam-like pixel scale
    return osys

osys = make_optical_system()

# --- numpy backend ---
poppy.conf.use_rust = False
poppy.accel_math.update_math_settings()
t0 = time.perf_counter()
psf_numpy = osys.calc_psf(wavelength=2e-6)
t_numpy = time.perf_counter() - t0
print(f'numpy  : {t_numpy*1000:.1f} ms')

# --- rust backend ---
poppy.conf.use_rust = True
poppy.accel_math.update_math_settings()
t0 = time.perf_counter()
psf_rust = osys.calc_psf(wavelength=2e-6)
t_rust = time.perf_counter() - t0
print(f'rust   : {t_rust*1000:.1f} ms')
print(f'speedup: {t_numpy/t_rust:.2f}x')

In [ ]:
numpy_data = psf_numpy[0].data
rust_data  = psf_rust[0].data

diff = numpy_data - rust_data
max_diff = np.max(np.abs(diff))
rel_diff = max_diff / numpy_data.max()

print(f'Max absolute difference : {max_diff:.2e}')
print(f'Relative to PSF peak    : {rel_diff:.2e}')
print(f'Peak pixel (numpy)      : {numpy_data.max():.6f}')
print(f'Peak pixel (rust)       : {rust_data.max():.6f}')
print(f'Total flux (numpy)      : {numpy_data.sum():.6f}')
print(f'Total flux (rust)       : {rust_data.sum():.6f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

norm = colors.LogNorm(vmin=numpy_data.max() * 1e-6, vmax=numpy_data.max())

im0 = axes[0].imshow(numpy_data, norm=norm, cmap='inferno', origin='lower')
axes[0].set_title('numpy backend', fontsize=13)
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(rust_data, norm=norm, cmap='inferno', origin='lower')
axes[1].set_title('rust backend', fontsize=13)
plt.colorbar(im1, ax=axes[1])

# Difference — use symmetric log scale so we can see both positive and negative residuals
absdiff = np.abs(diff)
if absdiff.max() > 0:
    diff_norm = colors.LogNorm(vmin=max(absdiff.max() * 1e-4, 1e-20), vmax=absdiff.max())
    im2 = axes[2].imshow(absdiff, norm=diff_norm, cmap='magma', origin='lower')
else:
    im2 = axes[2].imshow(absdiff, cmap='magma', origin='lower')
axes[2].set_title(f'|numpy − rust|  (max={max_diff:.1e})', fontsize=13)
plt.colorbar(im2, ax=axes[2])

plt.suptitle('Circular aperture PSF @ 2μm — NIRCam-like detector', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 2. Radial profile comparison
Check that the Airy disk rings line up exactly.

In [ ]:
cy, cx = np.array(numpy_data.shape) // 2
profile_numpy = numpy_data[cy, :]
profile_rust  = rust_data[cy, :]
pixels = np.arange(len(profile_numpy)) - cx

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.semilogy(pixels, profile_numpy, label='numpy', lw=2)
ax1.semilogy(pixels, profile_rust,  label='rust',  lw=1.5, ls='--', color='tomato')
ax1.set_ylabel('Intensity')
ax1.set_title('Horizontal PSF profile through peak')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(pixels, profile_numpy - profile_rust, color='purple')
ax2.axhline(0, color='black', lw=0.8)
ax2.set_ylabel('numpy − rust')
ax2.set_xlabel('Pixel offset from center')
ax2.set_title('Residual')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Multi-wavelength broadband PSF
NIRCam F200W — 9 wavelength samples across the bandpass.

In [ ]:
nlambda = 9
osys_bb = make_optical_system(fov_arcsec=3.0)

poppy.conf.use_rust = False
poppy.accel_math.update_math_settings()
t0 = time.perf_counter()
psf_bb_numpy = osys_bb.calc_psf(wavelength=2e-6, nlambda=nlambda)
t_bb_numpy = time.perf_counter() - t0

poppy.conf.use_rust = True
poppy.accel_math.update_math_settings()
t0 = time.perf_counter()
psf_bb_rust = osys_bb.calc_psf(wavelength=2e-6, nlambda=nlambda)
t_bb_rust = time.perf_counter() - t0

bb_diff = np.abs(psf_bb_numpy[0].data - psf_bb_rust[0].data)

print(f'Broadband ({nlambda} wavelengths)')
print(f'  numpy  : {t_bb_numpy:.2f} s')
print(f'  rust   : {t_bb_rust:.2f} s')
print(f'  speedup: {t_bb_numpy/t_bb_rust:.2f}x')
print(f'  max |diff|: {bb_diff.max():.2e}')

## 4. Toggle Rust on/off at runtime
Confirm `poppy.conf.use_rust` works as expected.

In [ ]:
for use_rust, label in [(False, 'numpy'), (True, 'rust')]:
    poppy.conf.use_rust = use_rust
    poppy.accel_math.update_math_settings()
    active = poppy.accel_math._USE_RUST
    print(f'conf.use_rust={use_rust}  →  _USE_RUST={active}  ({label})')

# Leave rust enabled
poppy.conf.use_rust = True
poppy.accel_math.update_math_settings()